# LongDocURL — Answer 覆盖率统计

**目标：** 统计用 answer 文本在 evidence page 内精确匹配，能定位到 chunk 的比例。  
同时统计：answer 长度分布、问题类型分布、表格/图片问题占比。

> 直接在 Colab / 本地 Jupyter 运行，Cell 顺序执行即可。


## 0. 安装依赖

In [ ]:
!pip install datasets rank_bm25 matplotlib pandas -q


## 1. 加载数据集 & 探查结构

In [ ]:
from datasets import load_dataset

print("Loading LongDocURL train split...")
ds = load_dataset("longdocurl/LongDocURL", split="train")
print(f"Total rows : {len(ds)}")
print(f"Columns    : {ds.column_names}")


In [ ]:
# 打印前 3 条，确认字段名和内容
for i in range(min(3, len(ds))):
    row = ds[i]
    print(f"{'='*60}  Sample {i}  {'='*60}")
    for k, v in row.items():
        if hasattr(v, 'size'):          # PIL Image
            print(f"  [{k}]  Image {v.size}")
        elif isinstance(v, list):
            print(f"  [{k}]  list[{len(v)}]  first={repr(str(v[0])[:120]) if v else '[]'}")
        elif isinstance(v, str):
            print(f"  [{k}]  {repr(v[:200])}")
        else:
            print(f"  [{k}]  {type(v).__name__} = {repr(str(v)[:100])}")
    print()


## 2. 配置字段名（根据上面的输出修改）

In [ ]:
# ── 根据 Cell 3 的输出修改这里 ──────────────────────────────
FIELD_QUESTION        = "question"
FIELD_ANSWER          = "answer"
FIELD_EVIDENCE_PAGES  = "evidence_pages"   # list[int], 1-indexed
FIELD_DOC_ID          = "doc_id"
FIELD_QTYPE           = "question_type"    # 若不存在改为 None
FIELD_PAGE_TEXTS      = "page_texts"       # list[str]，每页文本；若不存在改为 None
# ────────────────────────────────────────────────────────────

# 快速验证字段是否存在
for f in [FIELD_QUESTION, FIELD_ANSWER, FIELD_EVIDENCE_PAGES, FIELD_DOC_ID]:
    assert f in ds.column_names, f"字段 '{f}' 不存在，请修改 CONFIG"
    
optional = {FIELD_QTYPE: "question_type", FIELD_PAGE_TEXTS: "page_texts"}
for f, name in optional.items():
    if f and f not in ds.column_names:
        print(f"⚠️  可选字段 '{f}' 不存在，相关统计会跳过")
        
print("✅ 字段配置验证通过")


## 3. 工具函数

In [ ]:
import re, string
from collections import Counter, defaultdict

def normalize(text: str) -> str:
    """小写 + 去标点 + 合并空白（宽松匹配用）"""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()

def answer_in_text(answer: str, text: str) -> bool:
    """严格匹配：answer 是否出现在 text 中（子串）"""
    return answer.lower().strip() in text.lower()

def answer_in_text_loose(answer: str, text: str) -> bool:
    """宽松匹配：归一化后做子串检查"""
    return normalize(answer) in normalize(text)

def answer_token_len(answer: str) -> int:
    return len(answer.strip().split())

print("✅ 工具函数就绪")


## 4. Answer 长度分布

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

answers = [row[FIELD_ANSWER] for row in ds]
lengths = [answer_token_len(a) for a in answers]

length_series = pd.Series(lengths)
print(length_series.describe())
print()

bins = [0,1,2,3,5,10,20,50,999]
labels = ["1","2","3","4-5","6-10","11-20","21-50","50+"]
cats = pd.cut(length_series, bins=bins, labels=labels)
print("Answer token 长度分布：")
print(cats.value_counts().sort_index())

plt.figure(figsize=(8,3))
length_series.clip(upper=50).hist(bins=50)
plt.title("Answer token length distribution (clipped at 50)")
plt.xlabel("tokens"); plt.ylabel("count")
plt.tight_layout(); plt.show()


## 5. 问题类型分布

In [ ]:
if FIELD_QTYPE and FIELD_QTYPE in ds.column_names:
    qtypes = [row[FIELD_QTYPE] for row in ds]
    cnt = Counter(qtypes)
    print("问题类型分布：")
    for t, n in cnt.most_common():
        print(f"  {t:30s}  {n:6d}  ({n/len(ds)*100:.1f}%)")
else:
    print("⚠️  question_type 字段不存在，跳过此步骤")


## 6. Answer 精确匹配覆盖率统计

**逻辑：**
1. 取该条样本的 evidence page（页码）
2. 在这些页的文本中搜索 answer 是否出现（子串）
3. 统计"能匹配到"的比例

> 如果 `page_texts` 字段不存在，此 cell 会提示跳过。


In [ ]:
if not (FIELD_PAGE_TEXTS and FIELD_PAGE_TEXTS in ds.column_names):
    print("⚠️  page_texts 字段不存在，无法做 answer-in-page 精确匹配")
    print("   请确认数据集中存储页面文本的字段名，修改 FIELD_PAGE_TEXTS 后重跑")
else:
    results = []
    
    for row in ds:
        answer       = row[FIELD_ANSWER]
        evidence_pgs = row[FIELD_EVIDENCE_PAGES]   # list[int], 1-indexed
        page_texts   = row[FIELD_PAGE_TEXTS]        # list[str]
        qtype        = row.get(FIELD_QTYPE, "unknown") if FIELD_QTYPE else "unknown"
        
        # 取 evidence pages 的文本（页码转 0-indexed）
        ev_texts = []
        for pg in evidence_pgs:
            idx = int(pg) - 1  # 若已是 0-indexed 改成 int(pg)
            if 0 <= idx < len(page_texts):
                ev_texts.append(page_texts[idx])
        
        combined = " ".join(ev_texts)
        
        strict = answer_in_text(answer, combined)
        loose  = answer_in_text_loose(answer, combined)
        tok_len = answer_token_len(answer)
        
        results.append({
            "strict": strict,
            "loose":  loose,
            "tok_len": tok_len,
            "qtype":  qtype,
            "short_answer": tok_len <= 3,
        })
    
    df = pd.DataFrame(results)
    total = len(df)
    
    print(f"总样本数：{total}")
    print(f"严格匹配覆盖率：{df['strict'].mean()*100:.1f}%  ({df['strict'].sum()})")
    print(f"宽松匹配覆盖率：{df['loose'].mean()*100:.1f}%  ({df['loose'].sum()})")
    print()
    
    # 按问题类型拆分
    if FIELD_QTYPE and FIELD_QTYPE in ds.column_names:
        print("── 按问题类型拆分（严格匹配）──")
        g = df.groupby("qtype")["strict"].agg(["sum","count","mean"])
        g.columns = ["matched","total","coverage"]
        g["coverage"] = g["coverage"].map("{:.1%}".format)
        print(g.sort_values("total", ascending=False).to_string())
        print()
    
    # 按 answer 长度拆分
    print("── 按 answer token 长度拆分（严格匹配）──")
    bins   = [0,1,2,3,5,10,20,999]
    labels = ["1","2","3","4-5","6-10","11-20","21+"]
    df["len_bin"] = pd.cut(df["tok_len"], bins=bins, labels=labels)
    g2 = df.groupby("len_bin")["strict"].agg(["sum","count","mean"])
    g2.columns = ["matched","total","coverage"]
    g2["coverage"] = g2["coverage"].map("{:.1%}".format)
    print(g2.to_string())


## 7. BM25 chunk 级定位覆盖率

将 evidence page 文本切成 chunk（按段落），用 BM25 以 answer 为 query 检索，
看 top-1 chunk 内是否包含 answer。这模拟了"弱监督正样本定位"的真实精度。

> 依赖：`rank_bm25`（已在 Cell 0 安装）


In [ ]:
from rank_bm25 import BM25Okapi

def split_chunks(text: str, min_len: int = 20) -> list[str]:
    """按段落切分，过滤过短段落"""
    paras = re.split(r"\n{2,}", text.strip())
    return [p.strip() for p in paras if len(p.strip().split()) >= min_len]

if not (FIELD_PAGE_TEXTS and FIELD_PAGE_TEXTS in ds.column_names):
    print("⚠️  page_texts 字段不存在，跳过 BM25 chunk 级统计")
else:
    bm25_results = []
    
    for row in ds:
        answer       = row[FIELD_ANSWER]
        evidence_pgs = row[FIELD_EVIDENCE_PAGES]
        page_texts   = row[FIELD_PAGE_TEXTS]
        qtype        = row.get(FIELD_QTYPE, "unknown") if FIELD_QTYPE else "unknown"
        
        ev_texts = []
        for pg in evidence_pgs:
            idx = int(pg) - 1
            if 0 <= idx < len(page_texts):
                ev_texts.append(page_texts[idx])
        
        combined = " ".join(ev_texts)
        chunks = split_chunks(combined)
        
        if not chunks:
            bm25_results.append({"found": False, "n_chunks": 0, "qtype": qtype})
            continue
        
        # BM25 检索：query = answer tokens
        tokenized = [c.lower().split() for c in chunks]
        bm25 = BM25Okapi(tokenized)
        query_tokens = answer.lower().split()
        scores = bm25.get_scores(query_tokens)
        top_idx = scores.argmax()
        top_chunk = chunks[top_idx]
        
        found = answer_in_text(answer, top_chunk) or answer_in_text_loose(answer, top_chunk)
        bm25_results.append({"found": found, "n_chunks": len(chunks), "qtype": qtype})
    
    df_bm25 = pd.DataFrame(bm25_results)
    total = len(df_bm25)
    
    print(f"BM25 top-1 chunk 定位覆盖率：{df_bm25['found'].mean()*100:.1f}%  ({df_bm25['found'].sum()}/{total})")
    print(f"平均 chunk 数/evidence pages：{df_bm25['n_chunks'].mean():.1f}")
    print()
    
    if FIELD_QTYPE and FIELD_QTYPE in ds.column_names:
        print("── 按问题类型拆分（BM25 top-1）──")
        g = df_bm25.groupby("qtype")["found"].agg(["sum","count","mean"])
        g.columns = ["matched","total","coverage"]
        g["coverage"] = g["coverage"].map("{:.1%}".format)
        print(g.sort_values("total", ascending=False).to_string())


## 8. 结论 & 决策建议

运行完上面的 cell 后，根据数字填写下面的表格：

| 指标 | 数值 |
|---|---|
| 严格匹配覆盖率 | ? % |
| 宽松匹配覆盖率 | ? % |
| BM25 top-1 chunk 覆盖率 | ? % |
| answer token ≤ 3 的比例 | ? % |
| 表格/图片类问题占比 | ? % |

**决策阈值：**
- BM25 chunk 覆盖率 ≥ 60% → 可以直接用精确匹配 + BM25 构建训练数据
- BM25 chunk 覆盖率 40–60% → 需要用 answer sentence 策略 + 接受噪声
- BM25 chunk 覆盖率 < 40% → 必须引入 VLM 打分补充正样本，成本较高
